# Module 7 — Unsupervised Learning
## In-Class Practice — Solution Notebook

This notebook provides **one possible solution** to the in-class practice
for Module 7. Your own plots and exact metric values may differ slightly
depending on implementation details, but the overall structure and trends
should be similar.

In [ ]:
# Step 0 — Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

plt.rcParams['figure.figsize'] = (6, 5)
%matplotlib inline

---
## Part A — Dataset 1: Numeric Blobs

In [ ]:
# Fixed Dataset 1 generation
X1, y1_true = make_blobs(
    n_samples=500,
    centers=4,
    n_features=5,
    cluster_std=[1.0, 1.4, 1.2, 0.8],
    random_state=42
)

df1 = pd.DataFrame(X1, columns=[f'feat_{i}' for i in range(X1.shape[1])])
df1['true_label'] = y1_true
print(df1.shape)
df1.head()

### A.1 — Quick Exploration & Visualization

In [ ]:
print(df1.describe())

# Scatter without labels
plt.scatter(df1['feat_0'], df1['feat_1'], alpha=0.5)
plt.xlabel('feat_0')
plt.ylabel('feat_1')
plt.title('Dataset 1: feat_0 vs feat_1 (no labels)')
plt.show()

# Scatter coloured by true_label
for lab in sorted(df1['true_label'].unique()):
    sub = df1[df1['true_label'] == lab]
    plt.scatter(sub['feat_0'], sub['feat_1'], alpha=0.7, label=f'class {lab}')
plt.xlabel('feat_0')
plt.ylabel('feat_1')
plt.title('Dataset 1: feat_0 vs feat_1 coloured by true_label')
plt.legend()
plt.show()

### A.2 — Standardize Features

In [ ]:
X1_num = df1[[c for c in df1.columns if c.startswith('feat_')]].values
scaler1 = StandardScaler()
X1_scaled = scaler1.fit_transform(X1_num)

print('Mean (rounded):', np.round(X1_scaled.mean(axis=0), 3))
print('Std  (rounded):', np.round(X1_scaled.std(axis=0), 3))

### A.3 — K-Means for Multiple k

In [ ]:
k_values = list(range(2, 9))
inertias = []
silhouettes = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X1_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X1_scaled, labels)
    silhouettes.append(sil)

plt.plot(k_values, inertias, marker='o')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('K-Means Inertia vs k (Dataset 1)')
plt.show()

plt.plot(k_values, silhouettes, marker='o')
plt.xlabel('k')
plt.ylabel('Silhouette score')
plt.title('K-Means Silhouette vs k (Dataset 1)')
plt.show()

From these plots, a typical choice is **k = 4**, matching the underlying
number of blob centers and giving a high silhouette score with a clear
elbow in the inertia curve.

### A.4 — Final K-Means Model & Cluster Quality

In [ ]:
k_opt = 4
kmeans1 = KMeans(n_clusters=k_opt, random_state=42, n_init=10)
df1['kmeans_label'] = kmeans1.fit_predict(X1_scaled)

sil_km1 = silhouette_score(X1_scaled, df1['kmeans_label'])
db_km1 = davies_bouldin_score(X1_scaled, df1['kmeans_label'])
ch_km1 = calinski_harabasz_score(X1_scaled, df1['kmeans_label'])

print(f'K-Means (k={k_opt}) — silhouette: {sil_km1:.3f}')
print(f'K-Means (k={k_opt}) — Davies-Bouldin: {db_km1:.3f}')
print(f'K-Means (k={k_opt}) — Calinski-Harabasz: {ch_km1:.1f}')

# PCA projection
pca1 = PCA(n_components=2, random_state=42)
X1_pca = pca1.fit_transform(X1_scaled)

pca_df1 = pd.DataFrame(X1_pca, columns=['PC1', 'PC2'])
pca_df1['cluster'] = df1['kmeans_label']

for lab in sorted(pca_df1['cluster'].unique()):
    sub = pca_df1[pca_df1['cluster'] == lab]
    plt.scatter(sub['PC1'], sub['PC2'], alpha=0.7, label=f'cluster {lab}')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Dataset 1 — PCA projection coloured by K-Means clusters')
plt.legend()
plt.show()

### A.5 — Agglomerative (Hierarchical) Clustering

In [ ]:
agg1 = AgglomerativeClustering(n_clusters=k_opt, linkage='ward')
df1['agg_label'] = agg1.fit_predict(X1_scaled)

sil_agg1 = silhouette_score(X1_scaled, df1['agg_label'])
db_agg1 = davies_bouldin_score(X1_scaled, df1['agg_label'])
ch_agg1 = calinski_harabasz_score(X1_scaled, df1['agg_label'])

print(f'Agglomerative — silhouette: {sil_agg1:.3f}')
print(f'Agglomerative — Davies-Bouldin: {db_agg1:.3f}')
print(f'Agglomerative — Calinski-Harabasz: {ch_agg1:.1f}')

# PCA projection with agglomerative labels
pca_df1['agg_cluster'] = df1['agg_label']

for lab in sorted(pca_df1['agg_cluster'].unique()):
    sub = pca_df1[pca_df1['agg_cluster'] == lab]
    plt.scatter(sub['PC1'], sub['PC2'], alpha=0.7, label=f'cluster {lab}')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Dataset 1 — PCA projection coloured by Agglomerative clusters')
plt.legend()
plt.show()

**Summary for Dataset 1:**
- Both K-Means and Agglomerative (Ward) usually perform well on this kind of
  approximately spherical blob data.
- Their silhouette scores and CH indices are typically high and quite close.
- Small differences come from how each method optimizes its objective, but
  qualitatively the clusters align with the 4 true groups.

---
## Part B — Dataset 2: Two Moons

In [ ]:
# Fixed Dataset 2 generation
X2, y2_true = make_moons(n_samples=400, noise=0.08, random_state=42)
df2 = pd.DataFrame(X2, columns=['x1', 'x2'])
df2['true_label'] = y2_true
df2.head()

### B.1 — Visualize the Two Moons

In [ ]:
# Scatter without labels
plt.scatter(df2['x1'], df2['x2'], alpha=0.5)
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Dataset 2: Two Moons (no labels)')
plt.show()

# Scatter coloured by true_label
for lab in sorted(df2['true_label'].unique()):
    sub = df2[df2['true_label'] == lab]
    plt.scatter(sub['x1'], sub['x2'], alpha=0.7, label=f'class {lab}')
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Dataset 2: Two Moons coloured by true_label')
plt.legend()
plt.show()

### B.2 — K-Means on the Moons

In [ ]:
X2_num = df2[['x1', 'x2']].values
scaler2 = StandardScaler()
X2_scaled = scaler2.fit_transform(X2_num)

kmeans2 = KMeans(n_clusters=2, random_state=42, n_init=10)
df2['kmeans_label'] = kmeans2.fit_predict(X2_scaled)

sil_km2 = silhouette_score(X2_scaled, df2['kmeans_label'])
db_km2 = davies_bouldin_score(X2_scaled, df2['kmeans_label'])
ch_km2 = calinski_harabasz_score(X2_scaled, df2['kmeans_label'])

print(f'K-Means (k=2) — silhouette: {sil_km2:.3f}')
print(f'K-Means (k=2) — Davies-Bouldin: {db_km2:.3f}')
print(f'K-Means (k=2) — Calinski-Harabasz: {ch_km2:.1f}')

# Plot
for lab in sorted(df2['kmeans_label'].unique()):
    sub = df2[df2['kmeans_label'] == lab]
    plt.scatter(sub['x1'], sub['x2'], alpha=0.7, label=f'cluster {lab}')
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Dataset 2 — K-Means clustering (k=2)')
plt.legend()
plt.show()

K-Means tries to draw a **linear boundary** and often cuts through one or both
moons, so clusters do **not** align well with the true curved shapes.

### B.3 — DBSCAN on the Moons

In [ ]:
# Initial DBSCAN parameters (these work reasonably for this dataset)
dbscan2 = DBSCAN(eps=0.35, min_samples=5)
db_labels2 = dbscan2.fit_predict(X2_scaled)
df2['dbscan_label'] = db_labels2

unique_labels = np.unique(db_labels2)
print('Unique DBSCAN labels:', unique_labels)

n_noise = np.sum(db_labels2 == -1)
print('Number of noise points:', n_noise)

# Metrics on non-noise points if possible
mask_core = db_labels2 != -1
if mask_core.sum() > 0 and len(np.unique(db_labels2[mask_core])) > 1:
    sil_db2 = silhouette_score(X2_scaled[mask_core], db_labels2[mask_core])
    db_db2 = davies_bouldin_score(X2_scaled[mask_core], db_labels2[mask_core])
    ch_db2 = calinski_harabasz_score(X2_scaled[mask_core], db_labels2[mask_core])
    print(f'DBSCAN — silhouette (core/non-noise): {sil_db2:.3f}')
    print(f'DBSCAN — Davies-Bouldin (core/non-noise): {db_db2:.3f}')
    print(f'DBSCAN — Calinski-Harabasz (core/non-noise): {ch_db2:.1f}')
else:
    sil_db2 = np.nan
    db_db2 = np.nan
    ch_db2 = np.nan
    print('DBSCAN metrics not defined (too few non-noise clusters).')

# Plot DBSCAN clusters
plt.figure()
for lab in unique_labels:
    mask = db_labels2 == lab
    if lab == -1:
        plt.scatter(X2[mask, 0], X2[mask, 1], c='k', marker='x', label='noise')
    else:
        plt.scatter(X2[mask, 0], X2[mask, 1], alpha=0.7, label=f'cluster {lab}')
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Dataset 2 — DBSCAN clustering')
plt.legend()
plt.show()

With a suitable choice of `eps` and `min_samples`, DBSCAN recovers two
curved clusters that align much better with the true moon shapes, and
may mark a few borderline points as noise.

---
## Part C — Overall Reflection (Example Answer)


In [ ]:
reflection = """
- On Dataset 1 (5D blobs), both K-Means and Agglomerative (Ward) produce
  compact, well-separated clusters with similar internal metrics.
- On Dataset 2 (two moons), K-Means struggles because it assumes roughly
  convex/spherical clusters and uses a straight decision boundary.
- DBSCAN, by contrast, can follow the curved manifold of each moon and
  label low-density border points as noise.
- Internal metrics (silhouette, Davies–Bouldin, Calinski–Harabasz) are
  helpful but must be interpreted together with plots — especially when
  there is noise or non-convex geometry.
- PCA was most useful on Dataset 1 as a visualization tool to see the
  separation between clusters in 2D.
"""
print(reflection)